In [16]:
from time import sleep

import pandas as pd

from IPython.display import display

from pulao.constant import SwingPointLevel, SwingPointType
from pulao.trend import TrendManager
from pulao.swing import SwingManager
from pulao.bar import SBar, SBarManager, CBarManager, CBar
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import BarData

import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pl.read_csv("../dataset/I8888.XDCE_60m.csv", try_parse_dates=True)
df = df.head(1000)  # test

sbar_list = []
columns = df.columns

for idx, row in enumerate(df.iter_rows()):
    row_dict = dict(zip(columns, row))
    # datetime,open,close,high,low,volume,money,open_interest,signal
    datetime = row_dict["datetime"]
    open = row_dict["open"]
    close = row_dict["close"]
    high = row_dict["high"]
    low = row_dict["low"]
    volume = row_dict["volume"]
    money = row_dict["money"]
    open_interest = row_dict["open_interest"]

    bar = BarData(
        gateway_name="ctp-test",
        symbol="i8888",
        exchange=Exchange.SHFE,
        interval=Interval.MINUTE,
        datetime=datetime,
        open_price=open,
        close_price=close,
        high_price=high,
        low_price=low,
        volume=volume,
        open_interest=open_interest,
        turnover=money,
    )
    sbar = SBar(symbol=bar.symbol, exchange=bar.exchange.value, interval=bar.interval.value,datetime=datetime,turnover=money,open_price=open,close_price=close,high_price=high,low_price=low,volume=volume)

    sbar_list.append(sbar)
# 模拟行情数据接收
sbar_manager = SBarManager()
cbar_manager = CBarManager(sbar_manager=sbar_manager)
swing_manager = SwingManager(cbar_manager=cbar_manager)
trend_manager = TrendManager(swing_manager=swing_manager)

# 流入模拟数据
for sbar in sbar_list:
    sbar_manager.append(sbar)

#
# region 2. Plotly - Cbar
#
df_pandas2 = cbar_manager.df_cbar.to_pandas()
df_pandas2["datetime"] = pd.date_range("2025-01-01", periods=df_pandas2.shape[0], freq="h")
df_pandas2["open_price"] = df_pandas2["high_price"]
df_pandas2["close_price"] = df_pandas2["low_price"]

fig2 = make_subplots(
    rows=1, cols=1
)
fig2.add_trace(go.Candlestick(
    x=df_pandas2['datetime'],
    open=df_pandas2['open_price'],
    high=df_pandas2['high_price'],
    low=df_pandas2['low_price'],
    close=df_pandas2['close_price'],
    name='K线',
), row=1, col=1)

df_swing_point_low2 = df_pandas2[(df_pandas2['swing_point_type'] == "low") & (df_pandas2["swing_point_level"] == 2)]

fig2.add_trace(go.Scatter(
    x=df_swing_point_low2['datetime'],
    y=df_swing_point_low2['low_price'] * 0.995,   # 放在K线最低价下方一点，不遮挡蜡烛
    mode='markers',
    name='swing point low',
    marker=dict(
        symbol='triangle-up',
        size=14,
        color='#1E90FF',
    ),
    hovertemplate=
        "<b>波段低点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)


df_swing_point_high2 = df_pandas2[(df_pandas2['swing_point_type'] == "high") & (df_pandas2["swing_point_level"] == 2)]

fig2.add_trace(go.Scatter(
    x=df_swing_point_high2['datetime'],
    y=df_swing_point_high2['high_price'] * 1.005,  # 放在K线最高价上方一点
    mode='markers',
    name='swing point high',
    marker=dict(
        symbol='triangle-down',
        size=14,
        color='#FF4500',
    ),
    hovertemplate=
        "<b>波段高点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)


df_swing_point_low21 = df_pandas2[(df_pandas2['swing_point_type'] == "low") & (df_pandas2["swing_point_level"] == 1)]

fig2.add_trace(go.Scatter(
    x=df_swing_point_low21['datetime'],
    y=df_swing_point_low21['low_price'] * 0.995,   # 放在K线最低价下方一点，不遮挡蜡烛
    mode='markers',
    name='swing point low - 1',
    marker=dict(
        symbol='triangle-up',
        size=14,
        color='#333333',
    ),
    hovertemplate=
        "<b>波段低点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)


df_swing_point_high21 = df_pandas2[(df_pandas2['swing_point_type'] == "high") & (df_pandas2["swing_point_level"] == 1)]

fig2.add_trace(go.Scatter(
    x=df_swing_point_high21['datetime'],
    y=df_swing_point_high21['high_price'] * 1.005,  # 放在K线最高价上方一点
    mode='markers',
    name='swing point high - 1',
    marker=dict(
        symbol='triangle-down',
        size=14,
        color='#333333',
    ),
    hovertemplate=
        "<b>波段高点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)



# title = 'CBar Chart - K线合并处理'
title = 'CBar Chart - 分形重叠处理'
# title = 'CBar Chart - 次级别处理'
# title = 'CBar Chart - 连续同级别同向高低点处理'
fig2.update_layout(title=title,
    height=900,
    hovermode='x unified',    # X轴悬停联动虚线
)

fig2.update_xaxes(
    showgrid=False,
)

fig2.update_yaxes(
    showgrid=False,
)

fig2.show()


# endregion

CBar(index=343, start_index=452, end_index=452, high_price=882.8040161132812, low_price=878.2750244140625, swing_point_type='', swing_point_level=0, swing_point_level_origin=0) CBar(index=342, start_index=450, end_index=451, high_price=887.5180053710938, low_price=878.2750244140625, swing_point_type='', swing_point_level=0, swing_point_level_origin=0) SBar(index=453, symbol='i8888', exchange='SHFE', interval='1m', datetime=datetime.datetime(2024, 5, 8, 10, 0), volume=78325.0, turnover=6919264150.0, open_interest=0, open_price=881.789, high_price=884.489, low_price=875.529, close_price=877.654, swing_point_type=, swing_point_level=0, swing_point_level_origin=0, ema_short=881.5118008724238, ema_long=872.8774992890553)
CBar(index=343, start_index=452, end_index=452, high_price=882.8040161132812, low_price=878.2750244140625, swing_point_type='', swing_point_level=0, swing_point_level_origin=0) CBar(index=342, start_index=450, end_index=451, high_price=887.5180053710938, low_price=878.27502

Traceback (most recent call last):
  File "D:\PycharmProjects\pulao\pulao\events.py", line 25, in notify
    fn(event_type, payload)
    ~~^^^^^^^^^^^^^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_manager.py", line 39, in _on_sbar_created
    self.detect(payload)
    ~~~~~~~~~~~^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_manager.py", line 44, in detect
    self._agg_bar(sbar)
    ~~~~~~~~~~~~~^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_manager.py", line 77, in _agg_bar
    raise AssertionError(
        f"K线合并错误，出现了不应出现的情况 in {self.__class__.__name__}"
    )
AssertionError: K线合并错误，出现了不应出现的情况 in CBarManager
Traceback (most recent call last):
  File "D:\PycharmProjects\pulao\pulao\events.py", line 25, in notify
    fn(event_type, payload)
    ~~^^^^^^^^^^^^^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_manager.py", line 39, in _on_sbar_created
    self.detect(payload)
    ~~~~~~~~~~~^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_m

CBar(index=343, start_index=452, end_index=452, high_price=882.8040161132812, low_price=878.2750244140625, swing_point_type='', swing_point_level=0, swing_point_level_origin=0) CBar(index=342, start_index=450, end_index=451, high_price=887.5180053710938, low_price=878.2750244140625, swing_point_type='', swing_point_level=0, swing_point_level_origin=0) SBar(index=494, symbol='i8888', exchange='SHFE', interval='1m', datetime=datetime.datetime(2024, 5, 15, 23, 0), volume=43436.0, turnover=3735850550.0, open_interest=0, open_price=855.667, high_price=860.889, low_price=855.543, close_price=857.625, swing_point_type=, swing_point_level=0, swing_point_level_origin=0, ema_short=860.1551813420181, ema_long=865.7936098847271)
CBar(index=343, start_index=452, end_index=452, high_price=882.8040161132812, low_price=878.2750244140625, swing_point_type='', swing_point_level=0, swing_point_level_origin=0) CBar(index=342, start_index=450, end_index=451, high_price=887.5180053710938, low_price=878.2750

Traceback (most recent call last):
  File "D:\PycharmProjects\pulao\pulao\events.py", line 25, in notify
    fn(event_type, payload)
    ~~^^^^^^^^^^^^^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_manager.py", line 39, in _on_sbar_created
    self.detect(payload)
    ~~~~~~~~~~~^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_manager.py", line 44, in detect
    self._agg_bar(sbar)
    ~~~~~~~~~~~~~^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_manager.py", line 77, in _agg_bar
    raise AssertionError(
        f"K线合并错误，出现了不应出现的情况 in {self.__class__.__name__}"
    )
AssertionError: K线合并错误，出现了不应出现的情况 in CBarManager
Traceback (most recent call last):
  File "D:\PycharmProjects\pulao\pulao\events.py", line 25, in notify
    fn(event_type, payload)
    ~~^^^^^^^^^^^^^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_manager.py", line 39, in _on_sbar_created
    self.detect(payload)
    ~~~~~~~~~~~^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_m

CBar(index=343, start_index=452, end_index=452, high_price=882.8040161132812, low_price=878.2750244140625, swing_point_type='', swing_point_level=0, swing_point_level_origin=0) CBar(index=342, start_index=450, end_index=451, high_price=887.5180053710938, low_price=878.2750244140625, swing_point_type='', swing_point_level=0, swing_point_level_origin=0) SBar(index=531, symbol='i8888', exchange='SHFE', interval='1m', datetime=datetime.datetime(2024, 5, 23, 11, 0), volume=44852.0, turnover=4066951800.0, open_interest=0, open_price=911.922, high_price=912.252, low_price=905.023, close_price=908.568, swing_point_type=, swing_point_level=0, swing_point_level_origin=0, ema_short=906.0161276442093, ema_long=889.4704846364297)
CBar(index=343, start_index=452, end_index=452, high_price=882.8040161132812, low_price=878.2750244140625, swing_point_type='', swing_point_level=0, swing_point_level_origin=0) CBar(index=342, start_index=450, end_index=451, high_price=887.5180053710938, low_price=878.2750

...

Traceback (most recent call last):
  File "D:\PycharmProjects\pulao\pulao\events.py", line 25, in notify
    fn(event_type, payload)
    ~~^^^^^^^^^^^^^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_manager.py", line 39, in _on_sbar_created
    self.detect(payload)
    ~~~~~~~~~~~^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_manager.py", line 44, in detect
    self._agg_bar(sbar)
    ~~~~~~~~~~~~~^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_manager.py", line 77, in _agg_bar
    raise AssertionError(
        f"K线合并错误，出现了不应出现的情况 in {self.__class__.__name__}"
    )
AssertionError: K线合并错误，出现了不应出现的情况 in CBarManager
Traceback (most recent call last):
  File "D:\PycharmProjects\pulao\pulao\events.py", line 25, in notify
    fn(event_type, payload)
    ~~^^^^^^^^^^^^^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_manager.py", line 39, in _on_sbar_created
    self.detect(payload)
    ~~~~~~~~~~~^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_m

CBar(index=343, start_index=452, end_index=452, high_price=882.8040161132812, low_price=878.2750244140625, swing_point_type='', swing_point_level=0, swing_point_level_origin=0) CBar(index=342, start_index=450, end_index=451, high_price=887.5180053710938, low_price=878.2750244140625, swing_point_type='', swing_point_level=0, swing_point_level_origin=0) SBar(index=952, symbol='i8888', exchange='SHFE', interval='1m', datetime=datetime.datetime(2024, 8, 16, 15, 0), volume=103212.0, turnover=7227907450.0, open_interest=0, open_price=702.654, high_price=705.83, low_price=699.044, close_price=699.426, swing_point_type=, swing_point_level=0, swing_point_level_origin=0, ema_short=713.8842294279112, ema_long=733.1128290000936)
CBar(index=343, start_index=452, end_index=452, high_price=882.8040161132812, low_price=878.2750244140625, swing_point_type='', swing_point_level=0, swing_point_level_origin=0) CBar(index=342, start_index=450, end_index=451, high_price=887.5180053710938, low_price=878.2750

Traceback (most recent call last):
  File "D:\PycharmProjects\pulao\pulao\events.py", line 25, in notify
    fn(event_type, payload)
    ~~^^^^^^^^^^^^^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_manager.py", line 39, in _on_sbar_created
    self.detect(payload)
    ~~~~~~~~~~~^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_manager.py", line 44, in detect
    self._agg_bar(sbar)
    ~~~~~~~~~~~~~^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_manager.py", line 77, in _agg_bar
    raise AssertionError(
        f"K线合并错误，出现了不应出现的情况 in {self.__class__.__name__}"
    )
AssertionError: K线合并错误，出现了不应出现的情况 in CBarManager
Traceback (most recent call last):
  File "D:\PycharmProjects\pulao\pulao\events.py", line 25, in notify
    fn(event_type, payload)
    ~~^^^^^^^^^^^^^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_manager.py", line 39, in _on_sbar_created
    self.detect(payload)
    ~~~~~~~~~~~^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_m

CBar(index=343, start_index=452, end_index=452, high_price=882.8040161132812, low_price=878.2750244140625, swing_point_type='', swing_point_level=0, swing_point_level_origin=0) CBar(index=342, start_index=450, end_index=451, high_price=887.5180053710938, low_price=878.2750244140625, swing_point_type='', swing_point_level=0, swing_point_level_origin=0) SBar(index=986, symbol='i8888', exchange='SHFE', interval='1m', datetime=datetime.datetime(2024, 8, 23, 14, 0), volume=30826.0, turnover=2245074600.0, open_interest=0, open_price=729.336, high_price=730.714, low_price=726.863, close_price=727.771, swing_point_type=, swing_point_level=0, swing_point_level_origin=0, ema_short=728.0088967814553, ema_long=727.6073763804153)
CBar(index=343, start_index=452, end_index=452, high_price=882.8040161132812, low_price=878.2750244140625, swing_point_type='', swing_point_level=0, swing_point_level_origin=0) CBar(index=342, start_index=450, end_index=451, high_price=887.5180053710938, low_price=878.2750

Traceback (most recent call last):
  File "D:\PycharmProjects\pulao\pulao\events.py", line 25, in notify
    fn(event_type, payload)
    ~~^^^^^^^^^^^^^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_manager.py", line 39, in _on_sbar_created
    self.detect(payload)
    ~~~~~~~~~~~^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_manager.py", line 44, in detect
    self._agg_bar(sbar)
    ~~~~~~~~~~~~~^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_manager.py", line 77, in _agg_bar
    raise AssertionError(
        f"K线合并错误，出现了不应出现的情况 in {self.__class__.__name__}"
    )
AssertionError: K线合并错误，出现了不应出现的情况 in CBarManager
Traceback (most recent call last):
  File "D:\PycharmProjects\pulao\pulao\events.py", line 25, in notify
    fn(event_type, payload)
    ~~^^^^^^^^^^^^^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_manager.py", line 39, in _on_sbar_created
    self.detect(payload)
    ~~~~~~~~~~~^^^^^^^^^
  File "D:\PycharmProjects\pulao\pulao\bar\cbar_m

In [ ]:
sbar_manager.df_sbar
cbar_manager.df_cbar.reverse()